# 🛣️ Chapter 4 — Road Networks: Drivable vs. Walkable, Stats & Hubs
## Python for Urban Analysis — Ghana Edition

---

> **Professor's note:**
> *"A road network is a GRAPH. Thinking of streets as edges and intersections as nodes unlocks the entire apparatus of network science — centrality, shortest paths, connectivity, and resilience."*

### Learning Objectives
1. Download drivable and walkable networks from OSM
2. Compute basic network statistics (nodes, edges, density, circuity)
3. Calculate betweenness centrality to find network hubs
4. Classify roads by type (motorway, primary, residential, etc.)
5. Visualise networks with centrality-based colouring

### Ghana Context
Accra's road network tells a story of rapid, largely unplanned urbanisation. Major arterials (N1 Motorway, George Walker Bush Highway, Liberation Road) carry enormous loads, while the residential network is dense but poorly connected — a pattern typical of Sub-Saharan African cities.

## 4.0 — Install & Configure

In [ ]:
!pip install -q osmnx geopandas contextily networkx matplotlib mapclassify

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import osmnx as ox
import geopandas as gpd
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import contextily as cx
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────
STUDY_AREA       = 'Accra, Ghana'
CRS_WEB_MERCATOR = 'EPSG:3857'
CRS_UTM_ACCRA    = 'EPSG:32630'
DARK_BG          = '#0e1117'

DATA_DIR    = Path('data')
FIGURES_DIR = Path('outputs') / 'figures'
for d in [DATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'📍 Study area: {STUDY_AREA}')

## 4.1 — Download Drivable Network

We download the street network for cars using `network_type='drive'`. OSMnx converts the network data into a multi-directed graph (`MultiDiGraph`).

In [ ]:
print('🚗 Pulling DRIVABLE network ...')
G_drive = ox.graph_from_place(STUDY_AREA, network_type='drive')
print(f'   Nodes: {G_drive.number_of_nodes():,}')
print(f'   Edges: {G_drive.number_of_edges():,}')

## 4.2 — Download Walkable Network

We download the walkable network using `network_type='walk'`. This is typically much denser as it includes footpaths, pedestrian zones, and streets closed to cars.

In [ ]:
print('🚶 Pulling WALKABLE network ...')
G_walk = ox.graph_from_place(STUDY_AREA, network_type='walk')
print(f'   Nodes: {G_walk.number_of_nodes():,}')
print(f'   Edges: {G_walk.number_of_edges():,}')

## 4.3 — Basic Network Statistics

We compute structural statistics using `ox.stats.basic_stats()`. Key indicators include:
- **`n`**: Number of intersections (nodes)
- **`m`**: Number of street segments (edges)
- **`k_avg`**: Average node degree (average street connections per node)
- **`circuity_avg`**: Average ratio of actual street distance to straight-line distance. Higher means more winding streets.

In [ ]:
print('── Network Statistics (Drivable) ──')
stats_drive = ox.stats.basic_stats(G_drive)
for key in ['n', 'm', 'k_avg', 'edge_length_total', 'edge_length_avg', 'streets_per_node_avg', 'circuity_avg']:
    if key in stats_drive:
        val = stats_drive[key]
        if isinstance(val, float):
            print(f'   {key:30s}  {val:>12.2f}')
        else:
            print(f'   {key:30s}  {val:>12,}')

print('\n── Network Statistics (Walkable) ──')
stats_walk = ox.stats.basic_stats(G_walk)
for key in ['n', 'm', 'k_avg', 'edge_length_total', 'edge_length_avg', 'streets_per_node_avg', 'circuity_avg']:
    if key in stats_walk:
        val = stats_walk[key]
        if isinstance(val, float):
            print(f'   {key:30s}  {val:>12.2f}')
        else:
            print(f'   {key:30s}  {val:>12,}')

## 4.4 — Betweenness Centrality: Finding Network Hubs

Betweenness Centrality measures how often a node falls on the shortest path between all other pairs of nodes in the graph. Nodes with high betweenness centrality act as major routing hubs or traffic bottlenecks.

> **Note:** Calculating exact betweenness centrality on large graphs is computationally expensive ($O(N^3)$). In Colab, we sample $k=500$ nodes to approximate the scores quickly.

In [ ]:
print('🔬 Computing betweenness centrality (approximate) ...')

# Impute speeds and travel times
G_drive = ox.routing.add_edge_speeds(G_drive)
G_drive = ox.routing.add_edge_travel_times(G_drive)

# Convert parallel edges into a clean directed graph
D = ox.convert.to_digraph(G_drive, weight='travel_time')

# Compute centrality on a sample subset for speed
k_sample = min(500, D.number_of_nodes())
bc = nx.betweenness_centrality(D, weight='travel_time', k=k_sample)

# Set attribute on the original graph
nx.set_node_attributes(G_drive, values=bc, name='bc')

bc_sorted = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:10]
print('\n── Top 10 Network Hubs (Betweenness Centrality) ──')
for node_id, score in bc_sorted:
    lat = G_drive.nodes[node_id].get('y', 0)
    lon = G_drive.nodes[node_id].get('x', 0)
    print(f'   Node {node_id:>12}  BC={score:.4f}  ({lat:.4f}, {lon:.4f})')

## 4.5 — Road Type Classification

OSM classifies streets using the `highway` tag (e.g. `motorway`, `primary`, `secondary`, `residential`). We extract edges to a GeoDataFrame and group them.

In [ ]:
print('🛣️ Classifying roads by type ...')
edges = ox.graph_to_gdfs(G_drive, nodes=False, edges=True)

# Highway tag can occasionally be a list — flatten to the primary type
def flatten_highway(val):
    if isinstance(val, list):
        return val[0]
    return val

edges['road_type'] = edges['highway'].apply(flatten_highway)

print('\n── Road type breakdown ──')
road_counts = edges['road_type'].value_counts().head(15)
for rtype, n in road_counts.items():
    print(f'   {rtype:25s}  {n:>6}')

## 4.6 — Visualise Node Centrality

We map the nodes of the drivable graph using a colour gradient based on betweenness centrality. Nodes with higher values appear brighter, revealing key bottleneck zones.

In [ ]:
print('🎨 Creating network centrality map ...')
node_colors = ox.plot.get_node_colors_by_attr(G_drive, 'bc', cmap='plasma')

fig, ax = ox.plot.plot_graph(
    G_drive,
    node_color=node_colors,
    node_size=8,
    node_zorder=3,
    edge_color='#444444',
    edge_linewidth=0.5,
    bgcolor=DARK_BG,
    figsize=(14, 14),
    show=False,
    close=False,
)
ax.set_title(f'Road Network Centrality — {STUDY_AREA}', fontsize=16, fontweight='bold', color='white', pad=20)
fig.savefig(FIGURES_DIR / 'ch4_network_centrality.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 4.7 — Visualise Road Types on Basemap

We plot the street network with lines coloured by functional road type classifications.

In [ ]:
road_palette = {
    'motorway':      '#ff1744',
    'motorway_link': '#ff5252',
    'trunk':         '#ff9100',
    'trunk_link':    '#ffab40',
    'primary':       '#ffc400',
    'primary_link':  '#ffd740',
    'secondary':     '#69f0ae',
    'secondary_link':'#b9f6ca',
    'tertiary':      '#40c4ff',
    'tertiary_link': '#80d8ff',
    'residential':   '#b0bec5',
    'unclassified':  '#78909c',
    'service':       '#546e7a',
}

edges['color'] = edges['road_type'].map(road_palette).fillna('#37474f')
boundary = ox.geocode_to_gdf(STUDY_AREA)

fig, ax = plt.subplots(1, 1, figsize=(14, 14))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

edges.to_crs(CRS_WEB_MERCATOR).plot(
    ax=ax, color=edges['color'], linewidth=0.6, alpha=0.8
)

boundary.to_crs(CRS_WEB_MERCATOR).plot(
    ax=ax, facecolor='none', edgecolor='white', linewidth=2, zorder=10
)

cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=13)
ax.set_axis_off()
ax.set_title(f'Road Types — {STUDY_AREA}', fontsize=16, fontweight='bold', color='white', pad=20)

handles = [mpatches.Patch(color=c, label=t) for t, c in road_palette.items()]
ax.legend(
    handles=handles, loc='lower left', fontsize=7,
    facecolor='#1a1a2e', edgecolor='white', labelcolor='white',
    framealpha=0.9, ncol=2
)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch4_road_types.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 4.8 — Compare Drivable vs. Walkable Network

Let's compare walk and drive networks side by side to see the difference in urban connectivity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 12))
for i, (G, title) in enumerate([
    (G_drive, 'Drivable Network'),
    (G_walk,  'Walkable Network'),
]):
    ax = axes[i]
    fig.patch.set_facecolor(DARK_BG)
    ax.set_facecolor(DARK_BG)

    edges_g = ox.graph_to_gdfs(G, nodes=False, edges=True)
    edges_g.to_crs(CRS_WEB_MERCATOR).plot(
        ax=ax, color='#00e5ff' if i == 1 else '#ff9100',
        linewidth=0.4, alpha=0.7
    )
    boundary.to_crs(CRS_WEB_MERCATOR).plot(
        ax=ax, facecolor='none', edgecolor='white', linewidth=2
    )
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=13)
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight='bold', color='white')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch4_walk_vs_drive.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
# Save edge geometries
edges_path = DATA_DIR / 'road_edges.gpkg'
edges.to_file(edges_path, driver='GPKG')
print(f'💾 Road edges saved → {edges_path}')

## ✅ Chapter 4 Complete

### What you accomplished:
- Downloaded both drivable and walkable networks.
- Calculated network structural statistics and centrality hubs.
- Classified and mapped road networks by functional hierarchies.

### Next → Chapter 5: Composing the Urban Profile!